# 1. Set up

In [1]:
!pip install pandas
!pip install polars
!pip install scikit-learn
!pip install pyarrow
!pip install torch
!pip install xgboost category_encoders scikit-plot
!pip install pyproj
!pip install seaborn

# 2. Import necessary libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

from scipy.special import expit

from utils import *

# 3. Define global variables

we will be using march as the training month and april as the validation month

In [3]:
INPUT_X_PATH = "../data/output_data/data_training_may.csv"
INPUT_X_EVAL_PATH = "../data/output_data/data_validation_june.csv"
INPUT_Y_EVAL_PATH = "../data/output_data/data_validation_june_y.csv"
INPUT_Y_PATH = "../data/output_data/data_training_may_y.csv"

INPUT_FINAL_DF_PATH  = "../data/output_data/final_df_may.csv"
INPUT_FINAL_DF_EVAL_PATH  = "../data/output_data/final_df_eval_june.csv"


In [4]:
OUTPUT_FINAL_DF_PATH = "../data/output_data/final_df_non_supervised_may.csv"
OUTPUT_FINAL_DF_EVAL_PATH = "../data/output_data/final_df_eval_non_supervised_june.csv"

In [5]:
train_losses = list()
target_col = "target"

Let's define some colors for the viz we will be doing

In [6]:
ufd_orange = "#f26122"
ufd_blue = "#003865"
ufd_gray = "#cccccc"

# 4. Functions

# 5. Code

## 5.1. Data loading

We will load the data, both the training and the validation data

In [7]:
X = pd.read_csv(INPUT_X_PATH, sep=";")
y = pd.read_csv(INPUT_Y_PATH, sep=";")[target_col]
X_eval = pd.read_csv(INPUT_X_EVAL_PATH, sep=";")
y_eval = pd.read_csv(INPUT_Y_EVAL_PATH, sep=";")[target_col]

final_df = pd.read_csv(INPUT_FINAL_DF_PATH, sep=";")
final_df_eval = pd.read_csv(INPUT_FINAL_DF_EVAL_PATH, sep=";")

C:\Users\GAZARAGOZAG-local\AppData\Local\Temp\ipykernel_23184\1459858702.py:6: DtypeWarning: Columns (866) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(INPUT_FINAL_DF_PATH, sep=";")
C:\Users\GAZARAGOZAG-local\AppData\Local\Temp\ipykernel_23184\1459858702.py:7: DtypeWarning: Columns (1022) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df_eval = pd.read_csv(INPUT_FINAL_DF_EVAL_PATH, sep=";")


In [8]:
scaler = StandardScaler()
# Ahora escalar
X_scaled = scaler.fit_transform(X)
X_val_scaled = scaler.fit_transform(X_eval)

## 5.2. Non supervised learning

### 5.2.1. Modelling

#### DBSCAN cluster

First of all we will use the DBSCAN model. This algorithm is a cluster algorithm based on density. Groups the data that have sufficient neighbors and everything that does not go into a cluster is labeled as -1. Thus, the idea is no more than finiding outliers in the data points.

In [9]:
# Seteamos un radio de 3 alrededor de cada punto y un mínimo de 100 puntos requeridos dentro del vecindario para que un punto sea considerado un punto "core".
db = DBSCAN(eps=3, min_samples=100).fit(X_scaled)
# Identificamos outliers haciendo uso de la etiqueta -1
final_df["anomaly_dbscan"] = db.labels_ == -1

final_df["anomaly_dbscan"] = final_df["anomaly_dbscan"].replace({1: 0, 0: 1})

In [10]:
# final_df["anomaly_dbscan"].value_counts()

In [11]:
# # Calcular distancia promedio a todos los puntos core
# core_samples = db.core_sample_indices_
# if len(core_samples) > 0:
#     # Distancia mínima a puntos core
#     distances_to_core = pairwise_distances(X_scaled, X_scaled[core_samples])
#     min_distances_to_core = distances_to_core.min(axis=1)
    
#     # Normalizar distancias
#     dbscan_scores = MinMaxScaler().fit_transform(
#         min_distances_to_core.reshape(-1, 1)
#     ).flatten()
    
#     final_df["dbscan_distance_score"] = dbscan_scores

In [12]:
db_val = DBSCAN(eps=3, min_samples=100).fit(X_val_scaled)
final_df_eval["anomaly_dbscan"] = db_val.labels_ == -1
final_df_eval["anomaly_dbscan"] = final_df_eval["anomaly_dbscan"].replace({1: 0, 0: 1})

In [13]:
# final_df_eval["anomaly_dbscan"].value_counts()

In [14]:
# # Calcular distancia promedio a todos los puntos core
# core_samples = db_val.core_sample_indices_
# if len(core_samples) > 0:
#     # Distancia mínima a puntos core
#     distances_to_core = pairwise_distances(X_val_scaled, X_val_scaled[core_samples])
#     min_distances_to_core = distances_to_core.min(axis=1)
    
#     # Normalizar distancias
#     dbscan_scores = MinMaxScaler().fit_transform(
#         min_distances_to_core.reshape(-1, 1)
#     ).flatten()
    
#     final_df_eval["dbscan_distance_score"] = dbscan_scores

#### Isolation forest

Isolation forest es un modelo que trata de aislar los puntos de datos. Devuelve un -1 como un outlier y 1 como un punto "normal".

Isolation forest is a model that tries to aisle data points. It returns a -1 as an outlier and a 1 as a normal point. Thus, if we consider that a CUPS has a -1 prediction, it indicates that its metrics are far away from the general distribution.

In [15]:
iso = IsolationForest(contamination=0.01, random_state=42)
preds = iso.fit_predict(X_scaled)

# Extraer scores continuos
anomaly_scores = iso.decision_function(X_scaled)

iso_probabilities_sigmoid = expit(anomaly_scores * 5)  # Multiplica por factor para ajustar sensibilidad

final_df["iso_anomaly_scores"] = anomaly_scores
final_df["iso_prob_sigmoid"] = iso_probabilities_sigmoid
final_df["anomaly_iso"] = preds == -1


print("Información de Isolation Forest:")
print(f"Rango de scores: {anomaly_scores.min():.3f} a {anomaly_scores.max():.3f}")
print(f"Outliers detectados: {sum(preds == -1)}")
print(f"Rango de probabilidades sigmoid: {iso_probabilities_sigmoid.min():.3f} a {iso_probabilities_sigmoid.max():.3f}")

Información de Isolation Forest:
Rango de scores: -0.169 a 0.225
Outliers detectados: 2153
Rango de probabilidades sigmoid: 0.300 a 0.755


In [16]:
# iso = IsolationForest(contamination=0.01, random_state=42)
# preds = iso.fit_predict(X_scaled)
# final_df["anomaly_iso"] = preds == -1

In [17]:
iso_val = IsolationForest(contamination=0.01, random_state=42)
preds_val = iso_val.fit_predict(X_val_scaled)

# Extraer scores continuos
anomaly_scores_val = iso_val.decision_function(X_val_scaled)

iso_probabilities_sigmoid_val = expit(anomaly_scores_val * 5)  # Multiplica por factor para ajustar sensibilidad

final_df_eval["iso_anomaly_scores"] = anomaly_scores_val
final_df_eval["iso_prob_sigmoid"] = iso_probabilities_sigmoid_val
final_df_eval["anomaly_iso"] = preds_val == -1


print("Información de Isolation Forest:")
print(f"Rango de scores: {anomaly_scores.min():.3f} a {anomaly_scores.max():.3f}")
print(f"Outliers detectados: {sum(preds == -1)}")
print(f"Rango de probabilidades sigmoid: {iso_probabilities_sigmoid.min():.3f} a {iso_probabilities_sigmoid.max():.3f}")

Información de Isolation Forest:
Rango de scores: -0.169 a 0.225
Outliers detectados: 2153
Rango de probabilidades sigmoid: 0.300 a 0.755


In [18]:
# iso_val = IsolationForest(contamination=0.01, random_state=42)
# preds_val = iso_val.fit_predict(X_val_scaled)
# final_df_eval["anomaly_iso"] = preds_val == -1

#### Autoencoders

Now we are going to go with the approach of autoencoders. The idea here is to find big errors when the decoder tries to reconstruct the input. Then we will find anomalies within the magnitude of that error

In [19]:
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
X_tensor_val = torch.tensor(X_val_scaled, dtype=torch.float32)

# We create a PyTorch dataset that can be consumed by a DataLoader. TensorDataset is a way of evolving tensors
# as a simple dataset
train_data = TensorDataset(X_tensor)
val_data = TensorDataset(X_tensor_val)
# We create a data loader that divides the dataset in batches of 256 and shuffles the arrange randomly
loader = DataLoader(train_data, batch_size=256, shuffle=True)
loader_val = DataLoader(val_data, batch_size=256, shuffle=True)

In [20]:
model = Autoencoder(input_dim=X_tensor.shape[1])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=125, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=16, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=16, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=125, bias=True)
  )
)

In [21]:
model_val = Autoencoder(input_dim=X_tensor_val.shape[1])
optimizer_val = torch.optim.Adam(model_val.parameters(), lr=1e-3)

model_val.train()

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=125, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=16, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=16, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=125, bias=True)
  )
)

In [22]:
for epoch in range(30):
    model.train()
    running_loss = 0
    for batch in loader:
        # we take the batch[0] because TensorDataset only has one input tensor (without target)
        inputs = batch[0]
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        # we clear the gradients
        optimizer.zero_grad()
        # we calculate the gradients
        loss.backward()
        # we update the weights
        optimizer.step()
        running_loss += loss.item()

    train_losses.append(running_loss / len(loader))

    # Validation
    model.eval()

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_losses[-1]:.4f}")

Epoch 01 | Train Loss: 0.4794
Epoch 02 | Train Loss: 0.3482
Epoch 03 | Train Loss: 0.3185
Epoch 04 | Train Loss: 0.3028
Epoch 05 | Train Loss: 0.2872
Epoch 06 | Train Loss: 0.2738
Epoch 07 | Train Loss: 0.2656
Epoch 08 | Train Loss: 0.2557
Epoch 09 | Train Loss: 0.2478
Epoch 10 | Train Loss: 0.2441
Epoch 11 | Train Loss: 0.2392
Epoch 12 | Train Loss: 0.2349
Epoch 13 | Train Loss: 0.2299
Epoch 14 | Train Loss: 0.2251
Epoch 15 | Train Loss: 0.2248
Epoch 16 | Train Loss: 0.2214
Epoch 17 | Train Loss: 0.2173
Epoch 18 | Train Loss: 0.2146
Epoch 19 | Train Loss: 0.2176
Epoch 20 | Train Loss: 0.2081
Epoch 21 | Train Loss: 0.2086
Epoch 22 | Train Loss: 0.2051
Epoch 23 | Train Loss: 0.2012
Epoch 24 | Train Loss: 0.2017
Epoch 25 | Train Loss: 0.1987
Epoch 26 | Train Loss: 0.1968
Epoch 27 | Train Loss: 0.1994
Epoch 28 | Train Loss: 0.1907
Epoch 29 | Train Loss: 0.1920
Epoch 30 | Train Loss: 0.1930


We deactivate the gradient calculations (we save memory) and pass all the inputs to the model in order to get the reconstruction



In [23]:
with torch.no_grad():
    reconstructed_train = model(X_tensor)

In [24]:
train_losses_val = []

In [25]:
for epoch in range(30):
    model_val.train()
    running_loss = 0
    for batch in loader_val:
        # we take the batch[0] because TensorDataset only has one input tensor (without target)
        inputs = batch[0]
        outputs = model_val(inputs)
        loss = criterion(outputs, inputs)
        # we clear the gradients
        optimizer_val.zero_grad()
        # we calculate the gradients
        loss.backward()
        # we update the weights
        optimizer_val.step()
        running_loss += loss.item()

    train_losses_val.append(running_loss / len(loader_val))

    # Validation
    model.eval()

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_losses_val[-1]:.4f}")

Epoch 01 | Train Loss: 0.5087
Epoch 02 | Train Loss: 0.3767
Epoch 03 | Train Loss: 0.3425
Epoch 04 | Train Loss: 0.3224
Epoch 05 | Train Loss: 0.3075
Epoch 06 | Train Loss: 0.2962
Epoch 07 | Train Loss: 0.2867
Epoch 08 | Train Loss: 0.2814
Epoch 09 | Train Loss: 0.2729
Epoch 10 | Train Loss: 0.2672
Epoch 11 | Train Loss: 0.2620
Epoch 12 | Train Loss: 0.2585
Epoch 13 | Train Loss: 0.2538
Epoch 14 | Train Loss: 0.2461
Epoch 15 | Train Loss: 0.2439
Epoch 16 | Train Loss: 0.2392
Epoch 17 | Train Loss: 0.2360
Epoch 18 | Train Loss: 0.2352
Epoch 19 | Train Loss: 0.2319
Epoch 20 | Train Loss: 0.2279
Epoch 21 | Train Loss: 0.2270
Epoch 22 | Train Loss: 0.2233
Epoch 23 | Train Loss: 0.2211
Epoch 24 | Train Loss: 0.2172
Epoch 25 | Train Loss: 0.2148
Epoch 26 | Train Loss: 0.2140
Epoch 27 | Train Loss: 0.2108
Epoch 28 | Train Loss: 0.2102
Epoch 29 | Train Loss: 0.2080
Epoch 30 | Train Loss: 0.2080


In [26]:
with torch.no_grad():
    reconstructed_val = model_val(X_tensor_val)

In [27]:
mse_train = torch.mean((X_tensor - reconstructed_train) ** 2, dim=1)
final_df["reconstruction_error"] = mse_train

minmax_scaler = MinMaxScaler()
final_df["reconstruction_error_norm"] = minmax_scaler.fit_transform(
    final_df[["reconstruction_error"]]
).flatten()

# # O usar percentiles
# final_df["reconstruction_error_percentile"] = (
#     final_df["reconstruction_error"].rank() / len(final_df)
# )

threshold = final_df["reconstruction_error"].quantile(0.95)
final_df["anomaly_autoencoder"] = final_df["reconstruction_error"] > threshold

In [28]:
# mse_train = torch.mean((X_tensor - reconstructed_train) ** 2, dim=1)
# final_df["reconstruction_error"] = mse_train
# threshold = final_df["reconstruction_error"].quantile(0.99)
# final_df["anomaly_autoencoder"] = final_df["reconstruction_error"] > threshold

In [29]:
mse_train_eval = torch.mean((X_tensor_val - reconstructed_val) ** 2, dim=1)
final_df_eval["reconstruction_error"] = mse_train_eval

minmax_scaler_eval = MinMaxScaler()
final_df_eval["reconstruction_error_norm"] = minmax_scaler_eval.fit_transform(
    final_df_eval[["reconstruction_error"]]
).flatten()

# # O usar percentiles
# final_df["reconstruction_error_percentile"] = (
#     final_df["reconstruction_error"].rank() / len(final_df)
# )


threshold_val = final_df_eval["reconstruction_error"].quantile(0.95)
final_df_eval["anomaly_autoencoder"] = final_df_eval["reconstruction_error"] > threshold_val


In [30]:
# mse_val = torch.mean((X_tensor_val - reconstructed_val) ** 2, dim=1)
# final_df_eval["reconstruction_error"] = mse_val
# threshold_val = final_df_eval["reconstruction_error"].quantile(0.99)
# final_df_eval["anomaly_autoencoder"] = final_df_eval["reconstruction_error"] > threshold_val

#### Local outlier Factor

In [ ]:
# LOF proporciona scores continuos nativamente
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False)
lof_scores = lof.fit_predict(X_scaled)
lof_scores_continuous = -lof.negative_outlier_factor_  # Convertir a positivo

# Normalizar
lof_probabilities = MinMaxScaler().fit_transform(
    lof_scores_continuous.reshape(-1, 1)
).flatten()

final_df["anomaly_lof"] = lof_scores
final_df["lof_scores"] = lof_scores_continuous
final_df["lof_probabilities"] = lof_probabilities

In [ ]:
# LOF proporciona scores continuos nativamente
lof_eval = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False)
lof_scores_eval = lof_eval.fit_predict(X_val_scaled)
lof_scores_continuous_eval = -lof_eval.negative_outlier_factor_  # Convertir a positivo

# Normalizar
lof_probabilities_eval = MinMaxScaler().fit_transform(
    lof_scores_continuous_eval.reshape(-1, 1)
).flatten()

final_df_eval["anomaly_lof"] = lof_scores_eval
final_df_eval["lof_scores"] = lof_scores_continuous_eval
final_df_eval["lof_probabilities"] = lof_probabilities_eval

In [ ]:
# Cambiar valores en la columna anomaly_lof para final_df
# 1 -> 0, -1 -> 1
final_df["anomaly_lof"] = final_df["anomaly_lof"].replace({1: 0, -1: 1})

# Cambiar valores en la columna anomaly_lof para final_df_eval
# 1 -> 0, -1 -> 1
final_df_eval["anomaly_lof"] = final_df_eval["anomaly_lof"].replace({1: 0, -1: 1})

#### One class SVM

In [ ]:
# One-Class SVM con scores continuos
svm = OneClassSVM(nu=0.05, kernel='rbf', gamma='scale')
svm_preds = svm.fit_predict(X_scaled)
svm_scores = svm.decision_function(X_scaled)

# Convertir a probabilidades
svm_probabilities = expit(svm_scores)  # Función sigmoide

final_df["anomaly_svm"] = svm_preds
final_df["svm_scores"] = svm_scores
final_df["svm_probabilities"] = svm_probabilities

In [ ]:
final_df["svm_scores"].min(), final_df["svm_scores"].max()

(np.float64(-34.69458733730207), np.float64(49.61781533081714))

In [ ]:
# One-Class SVM con scores continuos
svm_eval = OneClassSVM(nu=0.05, kernel='rbf', gamma='scale')
svm_preds_eval = svm_eval.fit_predict(X_val_scaled)
svm_scores_eval = svm_eval.decision_function(X_val_scaled)

# Convertir a probabilidades
svm_probabilities_eval = expit(svm_scores_eval)  # Función sigmoide

final_df_eval["anomaly_svm"] = svm_preds_eval
final_df_eval["svm_scores"] = svm_scores_eval
final_df_eval["svm_probabilities"] = svm_probabilities_eval

In [ ]:
# Cambiar valores en la columna anomaly_lof para final_df
# 1 -> 0, -1 -> 1
final_df["anomaly_svm"] = final_df["anomaly_svm"].replace({1: 0, -1: 1})

# Cambiar valores en la columna anomaly_lof para final_df_eval
# 1 -> 0, -1 -> 1
final_df_eval["anomaly_svm"] = final_df_eval["anomaly_svm"].replace({1: 0, -1: 1})


### 5.2.2. Non supervised results

The idea here is to combine both anomalies, the one from DBSCAN and the one from the isolation forest. Those cups with most anomalies will be the strangest ones. Thus, we sum the anomalies columns and we set them as a possible fraud score.

In [ ]:
# final_df["anomaly_dbscan"] = final_df["anomaly_dbscan"] = 0
# final_df_eval["anomaly_dbscan"] = final_df_eval["anomaly_dbscan"] = 0

In [ ]:
final_df["fraude_score_no_supervisado"] = (
    (final_df["anomaly_dbscan"].astype(int) + final_df["anomaly_iso"].astype(int) + final_df["anomaly_autoencoder"].astype(int) + 
     final_df["anomaly_lof"].astype(int) + final_df["anomaly_svm"].astype(int))/5
)

final_df_eval["fraude_score_no_supervisado"] = (
    (final_df_eval["anomaly_dbscan"].astype(int) + final_df_eval["anomaly_iso"].astype(int) + final_df_eval["anomaly_autoencoder"].astype(int) + 
     final_df_eval["anomaly_lof"].astype(int) + final_df_eval["anomaly_svm"].astype(int))/5
)

## 5.3. Write the results

In [ ]:
final_df.to_csv(OUTPUT_FINAL_DF_PATH, sep=";", index=False)

final_df_eval.to_csv(OUTPUT_FINAL_DF_EVAL_PATH, sep=";", index=False)